In [87]:
import os

"""
Comet + Percolator：
    -r 指定的输出是peptide unique的; 其只包含target肽段
"""
def extract_bracketed_info(sequence):
    comet2pfind = {
        "15.9949": "Oxidation[M]",
        "42.0106": "Acetyl[Protein-Nterm]",
        "0.9840": "Deamidated[N]"
    }
    pfind_mod = ""  # 位置,mod;
    purified = []  # 构建净化序列（仅正常字母）
    i = 0
    n = len(sequence)
    while i < n:
        if sequence[i] == '[':
            # 1. 找到当前[对应的]的位置
            j = sequence.find(']', i + 1)
            if j == -1:  # 处理不完整的[]（跳过）
                i += 1
                continue
            # 2. 提取[xxx]中的内容
            bracketed_content = sequence[i+1:j]
            # 3. 计算[]的位置（核心修正逻辑）
            insert_index = len(purified)
            if insert_index == 0:
                # 情况1：[]在开头（插入点下标0，左侧无字母）→ 位置0
                position = 0
            else:
                position = insert_index
            # 4. 存储结果
            pfind_mod += f"{position},{comet2pfind[bracketed_content]};"
            # 5. 跳过当前[]，继续遍历
            i = j + 1
        elif sequence[i] == "C":  # 固定修饰
            purified.append(sequence[i])
            insert_index = len(purified)
            pfind_mod += f"{insert_index},Carbamidomethyl[C];"
            i += 1
        else:
            # 非[]内容：加入净化序列
            purified.append(sequence[i])
            i += 1
    purified_sequence = ''.join(purified)
    return purified_sequence, pfind_mod

# result_path = "D:/data/ms/MetaProteomics/humangut/1/peptide_results.txt"
result_path = "G:/CometWorkspace/humangut/peptide_results.txt"
comet_peptides = set()
cp2spec_name = dict()
comet_seqs = set()
decoy_num = 0
seq2pep = dict()

f = open(result_path, "r")
next(f,None)
for line in f:
    items = line.strip().split("\t")
    q_value = float(items[2])
    if q_value > 0.01:
        continue
    peptide = items[4][2:-2]
    if peptide[0]=='n':
        peptide = peptide[1:]
    seq, mod = extract_bracketed_info(peptide)
    peptide = seq+'\t'+mod

    # 判断是否为deocy肽段
    proteins = items[5:]
    flag = False
    for protein in proteins:
        if not protein.startswith("DECOY_"):
            flag = True
            break
    if not flag:
        decoy_num += 1
        continue

    if seq not in seq2pep:
        seq2pep[seq] = set()
    seq2pep[seq].add(peptide)
    comet_peptides.add(peptide)
    cp2spec_name[peptide] = items[0]
    comet_seqs.add(seq)
f.close()

print("鉴定到的target肽段数目:", len(comet_peptides))
print("鉴定到的target肽段序列数目:", len(comet_seqs))

鉴定到的target肽段数目: 122399
鉴定到的target肽段序列数目: 79833


In [56]:
"""
pFind
"""
pfind_res_path = "D:/pFindWorkspace/MetaProteomics/humangut-new/result/restricted/pFind-Filtered.spectra"

pfind_peptides = set()
pfind_seqs = set()

f = open(pfind_res_path,"r")
next(f,None)
for i,line in enumerate(f):
    items = line.split("\t")
    peptide = items[5]+"\t"+items[10]  # seq+mod
    proteins = items[12].split("/")[:-1]

    pfind_peptides.add(peptide)
    pfind_seqs.add(items[5])
f.close()

print("鉴定到的target肽段数目:", len(pfind_peptides))
print("鉴定到的target序列数目:", len(pfind_seqs))

鉴定到的target肽段数目: 108735
鉴定到的target序列数目: 93622


In [58]:
pfind_seq2peptide = dict()
for peptide in pfind_peptides:
    seq = peptide.split("\t")[0]
    if seq in pfind_seq2peptide:
        pfind_seq2peptide[seq].append(peptide)
    else:
        pfind_seq2peptide[seq] = [peptide]

In [81]:
peptides = comet_peptides - pfind_peptides
a = 0
b = 0
for i,peptide in enumerate(peptides):
    seq = peptide.split("\t")[0]
    if seq in pfind_seqs:
        b += 1
        pps = pfind_seq2peptide[seq]
        for ps in pps:
            if len(ps)==len(peptide):
                a += 1
    # if i>100:
    #     break
    # print(peptide, cp2spec_name[peptide])
    #     # break
print(len(peptides), a, b)

58839 4433 30573


In [35]:
# len(comet_seqs-pfind_seqs)
a = comet_seqs - pfind_seqs
num = 0
for i,aa in enumerate(a):
    if i<4:
        continue
    print(aa)
    break
print(len(a), num)

LVSDAAPTGCVYK
17182 0


In [89]:
len(pfind_seqs-comet_seqs)

30971

In [21]:
a = comet_peptides - pfind_peptides
num = 0
for aa in a:
    # if ";" in aa:
    #     num += 1
    if "Dea" in aa:
        num += 1
        print(aa)
        break
print(len(a), num)

SALGALQNRLEHTVANLDNVAENTQSAESR	23,Deamidated[N];
48701 1


In [85]:
len(pfind_peptides-comet_peptides)

31903

In [39]:
for peptide in comet_peptides:
    print(peptide)
    break

IPTLGTYYCGFNVNSDLFAGK	9,Carbamidomethyl[C];14,Deamidated[N];


In [75]:
a = 0
b = 0
for seq,pep_list in seq2pep.items():
    if len(pep_list)<2:
        continue
    a += 1
    mls = list()
    for pep in pep_list:
        mods = pep.split("\t")[-1]
        ml = list()
        for mod in mods.split(";"):
            if mod == "":
                break
            ml.append(mod.split(",")[1])
        ml = sorted(ml)
        mls.append(ml)
    mls = sorted(mls)
    k = 1
    for i in range(1,len(mls)):
        if mls[i] != mls[i-1]:
            k += 1
    if len(pep_list)-k:
        b += 1
print(a,b)

22947 7828


In [57]:
a = (1,2)
a.insert(3)

AttributeError: 'tuple' object has no attribute 'insert'